<a href="https://colab.research.google.com/github/jppeirce/DSC210-Foundations-of-Data-Science/blob/main/ClassNotebooks/09-unsupervised_learning/09-unsupervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9: Unsupervised Learning

**DSC 210 Foundations of Data Science**

References:
- [Hands-on Introduction to Data Science with Python](https://florian-huber.github.io/data_science_course/) (CC BY-NC-SA 4.0)
- [scikit-learn user guide](https://scikit-learn.org/stable/modules/clustering.html), on clustering and decomposition
- [Principal Component Analysis (PCA) Explained Visually with Zero Math](https://towardsdatascience.com/principal-component-analysis-pca-explained-visually-with-zero-math-1cbf392b9e7d/), *Towards Data Science* (the framing of Sections 8.2 and 8.4: variance as information, rotate then drop, and distances shrinking)

```
ASK  ->  GET  ->  EXPLORE  ->  [ MODEL ]  ->  COMMUNICATE
```

*Last major revision: 2026-10-26*

Everything so far has been description: what is in the data, what is wrong with it, what it looks like. Today we start **modeling**, and we begin with the half of modeling that has no answer key.

## Key Concepts

- Distinguish **supervised** from **unsupervised** learning by asking whether the data has labels
- State the **k-means objective** and carry out the algorithm by hand on a small dataset
- Explain why k-means needs `k` chosen in advance, and use an elbow plot to choose it
- Build a **dendrogram** by hand with single linkage, and read a cut as a clustering
- Describe how **DBSCAN** differs from k-means, and when its shape assumptions win
- Use **PCA** to reduce dimensions: explain why features must be scaled first, read a scree plot and the loadings, and say what is lost when components are dropped

---
## 1. Learning Without an Answer Key
---

Suppose a store hands you 2,240 customer records and asks: *"what kinds of customers do we have?"*

Notice what is missing from that question. Nobody has told you what the kinds *are*. There is no column labelled `customer_type` that you are trying to predict. There is no answer key against which your output can be graded.

**Definition.** In **supervised learning** the data includes a label (an answer) for each case, and the goal is to predict that label for new cases. In **unsupervised learning** there are no labels, and the goal is to find structure that was already there.

| | Supervised | Unsupervised |
| --- | --- | --- |
| Data has labels? | Yes | No |
| Typical question | "Will this customer respond?" | "What kinds of customers are there?" |
| How do you know you are right? | Compare predictions to held-out labels | You cannot, not directly |
| This course | Module 10 | Today |

When there is no answer key, **you cannot compute an accuracy**. A clustering is not right or wrong. That is uncomfortable, and it is also why the techniques in this module are so often paired with the domain expertise circle from Module 1.

Today's two families:

- **Clustering** groups similar cases together. (Sections 2 through 5.)
- **Dimension reduction** replaces many features with a few that carry most of the information. (Section 6.)

Both halves of that table live inside the same field. It is worth seeing where that field sits before we go further.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_ai_vs_ml_vs_deep_learning.png?raw=true" width="400">

*Machine learning is one region of a larger territory, and **both** supervised and unsupervised learning sit inside it. Deep learning, the innermost circle, is a family of methods we will not reach in this course.*

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_clustering_intro.png?raw=true" width="620">

*Clustering asks the machine to find groups that nobody labelled. The same points can be grouped more than one defensible way, which is exactly why judging the result is a human job.*

In [ ]:
# RUN-TOGETHER
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

---
## 2. k-means Clustering
---
### 2.1 The idea, and the objective

k-means splits the data into `k` groups, each represented by a **centroid** (the mean of the points assigned to it). It does this by trying to make every point close to its own centroid.

**Definition.** The **within-cluster sum of squares** (WCSS) of a clustering is

$$\text{WCSS} \;=\; \sum_{j=1}^{k} \; \sum_{x \,\in\, C_j} \left(x - c_j\right)^2$$

where $C_j$ is the $j$-th cluster and $c_j$ is its centroid. **k-means searches for the assignment that makes WCSS as small as possible.**

Read the formula in words before moving on: for each cluster, measure how far every point is from that cluster's centre, square it, and add everything up. A small WCSS means tight clusters.

The algorithm is a repeated two-step:

1. **Assign.** Put each point with the nearest centroid.
2. **Update.** Move each centroid to the mean of the points assigned to it.

Repeat until nothing changes. Each step can only lower WCSS or leave it alone, which is why the procedure stops.

The four panels below show one full run. **A**: centroids are placed. **B**: every point is assigned to its nearest centroid. **C**: each centroid moves to the mean of its members. **D**: the process has stopped changing.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_kmeans_sketch.png?raw=true" width="700">

#### **Activity 9.1 - k-means by hand**

Before we run any code, we will compute the algorithm by hand. Consider four points on a line:

$$x = 1,\; 2,\; 6,\; 7 \qquad k = 2$$

We will deliberately start with a **bad** pair of centroids, $c_1 = 1$ and $c_2 = 2$, both sitting in the same clump, to see whether k-means can recover.

Work down the table. For each point, compute the distance to each centroid and assign it to the nearer one. Then recompute each centroid as the mean of its members, and compute the WCSS.

**Iteration 1** ($c_1 = 1$, $c_2 = 2$)

| point | $\lvert x - c_1 \rvert$ | $\lvert x - c_2 \rvert$ | cluster |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 2 |  |  |  |
| 6 |  |  |  |
| 7 |  |  |  |

New $c_1$ = __________  New $c_2$ = __________  WCSS = __________

**Iteration 2** (use your new centroids)

| point | $\lvert x - c_1 \rvert$ | $\lvert x - c_2 \rvert$ | cluster |
| --- | --- | --- | --- |
| 1 |  |  |  |
| 2 |  |  |  |
| 6 |  |  |  |
| 7 |  |  |  |

New $c_1$ = __________  New $c_2$ = __________  WCSS = __________

**Iteration 3** (use your new centroids)

Did any point change cluster? If not, the algorithm has **converged** and you are done.

**Questions.**

**A.** How many iterations did it take? What told you to stop?

**B.** Write down your two WCSS values. What happened to WCSS from iteration 1 to iteration 2, and why could it never have gone up?

**C.** Now redo iteration 1 only, starting instead from $c_1 = 1$ and $c_2 = 7$. How many iterations would this start have needed? What does that tell you about the initial centroids?

#### **Class Example 9.1 - The same four points, in code**

Now we check our arithmetic. `KMeans` from scikit-learn expects a 2-D array, so our four numbers become four rows of one column.

In [ ]:
# RUN-TOGETHER
from sklearn.cluster import KMeans

X = np.array([[1.], [2.], [6.], [7.]])

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)

print('cluster labels :', km.labels_)
print('centroids      :', km.cluster_centers_.ravel())
print('WCSS (inertia_):', round(km.inertia_, 3))

Compare against your table. The labels may be swapped (cluster "0" and cluster "1" are arbitrary names), but the **centroids** and the **WCSS** should match what you computed by hand.

scikit-learn calls the WCSS `inertia_`. It is the same quantity from the definition above, and we will use it again in Section 3.

One detail worth noting: `n_init=10` runs the whole algorithm ten times from ten different random starts and keeps the best. Activity 9.1 part C is exactly why that argument exists.

Here is the whole run as a picture. Check each panel against the row of your table.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_kmeans_by_hand.png?raw=true" width="640">

Notice panel 2 to panel 3: the point at 2 changes sides. That single switch is what drops WCSS from 14 to 1.

### 2.2 k-means on real data: the penguins

Four points on was a nice toy problem. Here is the same algorithm on a dataset we have seen before.

The penguins dataset has four body measurements and, in a separate column, the species. We are going to **hide the species from the algorithm**, cluster on the measurements alone, and then reveal the species to see what k-means found on its own. 

In [ ]:
# RUN-TOGETHER
penguins = sns.load_dataset('penguins').dropna()

features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
sns.pairplot(penguins, vars=features, hue=None, diag_kind='hist')

### **Scale before K Means!**

K-means assigns points to clusters using Euclidean distance, so any feature with a larger numeric range dominates that distance. For example, if one feature is income (tens of thousands) and another is age (tens), income will drive nearly all the cluster assignments, regardless of which feature is actually more informative. Scaling puts features on a comparable footing so each contributes proportionally to the distance calculation and to the cluster centroids. Standardization (zero mean, unit variance) is the most common choice, and it also helps K-means converge more reliably and produce clusters that reflect the underlying structure of the data rather than the arbitrary units of measurement.

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler

X_pen = StandardScaler().fit_transform(penguins[features])   # scale first!

km_pen = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X_pen)
penguins['cluster'] = km_pen.labels_

print('cluster sizes:')
print(penguins['cluster'].value_counts().sort_index())

Now the reveal. Cross-tabulate the clusters against the species the algorithm never saw.

In [ ]:
# RUN-TOGETHER
print(pd.crosstab(penguins['cluster'], penguins['species']))

What can we read from this table?

- One cluster contains **119 Gentoo and nothing else**. k-means found that species perfectly, without being told it existed.
- Another is almost all Adelie; a third is mostly Chinstrap.
- The mistakes are all **Adelie confused with Chinstrap**, in both directions.

About 92% of penguins land in the "right" cluster, and the errors are not random. Adelie and Chinstrap are close in body size and differ mainly in bill length, while Gentoo are substantially bigger than both. So k-means did not find *species*. It found **size**, and size happens to identify Gentoo cleanly while separating the other two only partly.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_kmeans_penguins.png?raw=true" width="820">

> **Discuss.** The algorithm was given four measurements and no labels, and it recovered most of the species structure. Does that mean clustering "works"? Now suppose the three species had been identical in size but differed in colour, a feature not in our table. What would k-means have returned, and how would you have known it was wrong?

### 2.3 Where k-means fails

k-means minimizes squared distance to a center, and that single fact determines the shapes it can find. Clusters that are round and similarly sized: fine. Anything else: trouble.

To see the failure cleanly we use **deliberately constructed** data rather than a real dataset, because we want a shape chosen to break the assumption.

In [ ]:
# RUN-TOGETHER
from sklearn.datasets import make_moons

X_moons, _ = make_moons(n_samples=300, noise=0.06, random_state=210)
km_moons = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], color='gray', s=20, ax=axes[0])
axes[0].set_title('the data: two crescents')
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=km_moons.labels_, palette='viridis', s=20, ax=axes[1], legend=False)
axes[1].set_title('what k-means finds')
plt.show()

Your eye sees two crescents. k-means draws a straight line through both of them, because the only clustering it can express is "closest centre wins", and that always carves space into straight-edged regions.

**The three standing limitations of k-means:**

1. You must choose `k` in advance (Section 3).
2. The result depends on the initial centroids (Activity 9.1, part C).
3. It assumes clusters are round and comparably sized.

Section 5 introduces an algorithm with a different assumption, which handles the crescents easily and fails elsewhere. There is no clustering algorithm that always wins.

---
## 3. Choosing k
---

If `k` must be chosen in advance, how do we choose it?

The honest answer is that domain knowledge usually decides: a marketing team that can run three campaigns wants three segments. But there is a useful diagnostic, and it reuses the WCSS you computed by hand.

Run k-means for many values of `k` and plot WCSS against `k`, using the penguins again. WCSS always falls as `k` rises (more centroids means everything is nearer to one), so we are not looking for a minimum. We are looking for the point where the *improvement* stops being worth it: a bend, or **elbow**.

In [ ]:
# RUN-TOGETHER
ks = range(1, 9)
wcss = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_pen).inertia_ for k in ks]

for k, w in zip(ks, wcss):
    print(f'k = {k}   WCSS = {w:7.1f}')

sns.lineplot(x=list(ks), y=wcss, marker='o')
plt.title('Elbow plot: penguins, four scaled measurements')
plt.xlabel('k (number of clusters)')
plt.ylabel('WCSS (inertia)')
plt.xticks(list(ks))
plt.show()

#### **Activity 9.2 - Read the elbow**

**A.** Look at the printed WCSS values. Compute the *drop* at each step: from k=1 to k=2, from k=2 to k=3, from k=3 to k=4. Which drop is by far the largest?

**B.** Based on the drops alone, what value of `k` does the elbow suggest? We know the data contains **three** species. Does the elbow agree?

**C.** This disagreement is not a mistake in the plot. Using what you learned in Section 2.2 about *what* k-means actually separated, explain why the elbow points where it does.

**D.** WCSS at `k = 8` is lower than at any smaller `k`. Explain in one sentence why that does not make 8 the better choice. What would WCSS be if `k` equalled the number of penguins?

**E.** Now build the same elbow plot for the crescent data `X_moons`. Is there a clear elbow? What does its absence tell you?

In [ ]:
# FILL-IN  (Activity 9.2, Part D)
wcss_moons = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(____).inertia_ for k in ks]

sns.lineplot(x=list(ks), y=wcss_moons, marker='o')
plt.title('Elbow plot for the crescent data')
plt.xlabel('k'); plt.ylabel('WCSS')
plt.show()

---
## 4. Hierarchical Clustering
---

k-means asks you to commit to `k` before you see anything. **Hierarchical clustering** builds the entire family of clusterings at once, from "every point alone" up to "everything in one group", and lets you cut wherever you like afterwards.

The version we use is **agglomerative** (bottom-up):

1. Start with every point as its own cluster.
2. Merge the two **closest** clusters.
3. Repeat until one cluster remains, recording the distance at each merge.

Step 2 needs a definition of distance *between clusters*, not between points, and there is more than one reasonable choice. That choice is called the **linkage**.

**Definition.** Under **single linkage**, the distance between two clusters is the distance between their two *nearest* members:

$$d(A, B) \;=\; \min_{a \in A,\; b \in B} \; \lvert a - b \rvert$$

The record of merges is drawn as a **dendrogram**: a tree whose branch heights are the distances at which merges happened. Cutting the tree at a height gives you a clustering.

#### **Activity 9.3 - Build a dendrogram by hand**

Five points on a line:

$$A = 1, \quad B = 2, \quad C = 5, \quad D = 9, \quad E = 11$$

**Step 1. Fill in the distance matrix.** (It is symmetric, so only the upper triangle is needed.)

| | A | B | C | D | E |
| --- | --- | --- | --- | --- | --- |
| **A** | 0 |  |  |  |  |
| **B** | | 0 |  |  |  |
| **C** | | | 0 |  |  |
| **D** | | | | 0 |  |
| **E** | | | | | 0 |

**Step 2. Merge, four times.** Each round: find the smallest distance in the table, merge that pair, and record the height. Then recompute distances from the new cluster to everything else using **single linkage** (the minimum).

| merge # | clusters joined | height |
| --- | --- | --- |
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 4 |  |  |

**Step 3. Sketch the dendrogram** with height on the vertical axis.

**Questions.**

**A.** Cut your dendrogram at height 3.5. Which clusters do you get?

**B.** Cut it at height 1.5 instead. How many clusters now?

**C.** k-means needed `k` before starting. What did hierarchical clustering need instead, and when did you need to supply it?

**D.** Under single linkage, the distance from $\{A,B\}$ to $C$ was the *minimum* of $d(A,C)$ and $d(B,C)$. Suppose we had used the **maximum** instead (called complete linkage). Redo merge 2 and say whether anything changes.

#### **Class Example 9.2 - The same five points, in code**

In [ ]:
# RUN-TOGETHER
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

pts = np.array([[1.], [2.], [5.], [9.], [11.]])
labels = ['A', 'B', 'C', 'D', 'E']

Z = linkage(pts, method='single')

dendrogram(Z, labels=labels)
plt.title('Single-linkage dendrogram')
plt.ylabel('merge height (distance)')
plt.show()

print('merge heights:', [round(z[2], 2) for z in Z])
print('cut at 3.5 ->', fcluster(Z, 3.5, criterion='distance'))

Check the printed merge heights against your table, and the cut against your answer to question A.

Notice what the dendrogram gives you that k-means never did: **every** clustering, at once. You choose the number of clusters by choosing a height, after looking at the picture rather than before.

---
## 5. DBSCAN: Clustering by Density
---

k-means asks "which center is nearest?" DBSCAN asks a different question: **"is this point in a crowded neighbourhood?"**

DBSCAN stands for Density-Based Spatial Clustering of Applications with Noise. It is a popular unsupervised machine learning algorithm used to group data points that are closely packed together (high density) while marking isolated points as outliers or noise.

**Definition.** DBSCAN takes two parameters:

- `eps` ($\varepsilon$, "epsilon"): a radius defining what "nearby" means.
- `min_samples`: how many points must lie within that radius for a neighbourhood to count as crowded.

Every point is then labelled:

- a **core point** has at least `min_samples` points within `eps`;
- a **border point** is within `eps` of a core point but is not itself core;
- a **noise point** is neither.

Clusters are built by chaining core points together, so a cluster can be any shape at all: a crescent, a ring, a snake. And unlike k-means, DBSCAN is allowed to say *"this point belongs to nothing"*, which no version of k-means can do. 

Noise points are labeled as cluster -1.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_dbscan_sketch.png?raw=true" width="640">

*Core points sit in crowded neighbourhoods, border points touch a core point, and noise points sit alone. Clusters grow by chaining core points, which is why a cluster can bend around a curve.*

In [ ]:
# RUN-TOGETHER
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=0.25, min_samples=5).fit(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=km_moons.labels_, palette='viridis', s=20, ax=axes[0], legend=False)
axes[0].set_title('k-means, k = 2')
sns.scatterplot(x=X_moons[:, 0], y=X_moons[:, 1], hue=db.labels_, palette='viridis', s=20, ax=axes[1], legend=False)
axes[1].set_title('DBSCAN, eps = 0.25')
plt.show()

print('clusters found :', len(set(db.labels_)) - (1 if -1 in db.labels_ else 0))
print('points labelled noise:', (db.labels_ == -1).sum())

DBSCAN separates the crescents, and it was never told there were two. The number of clusters is an *output*, not an input.

The cost is that `eps` and `min_samples` now carry the difficulty that `k` used to. There is no free lunch: every clustering algorithm requires you to say what "similar" means, and each one asks in its own language.

## The No Free Lunch Theorem
> The No Free Lunch (NFL) theorem states that when performance is averaged across all possible problems, no single optimization or machine learning algorithm is universally superior. If an algorithm performs exceptionally well on one class of problems, it must pay for that advantage by performing worse on other problems

#### **Activity 9.4 - Tuning eps**

**A.** Predict: what happens to the number of clusters if `eps` is very small? Very large?

**B.** Run the cell below for `eps` values 0.1, 0.25, and 1.0 and record the number of clusters and the number of noise points for each.

**C.** Which of the three would you report, and why?

**D.** In one sentence: DBSCAN found the crescents and k-means could not. Name a dataset shape where you would expect the opposite.

In [ ]:
# FILL-IN  (Activity 9.4, Part B)
for eps in [0.1, 0.25, 1.0]:
    db = DBSCAN(eps=eps, min_samples=5).fit(X_moons)
    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    print(f'eps={eps:5}  clusters={n_clusters}  noise={(db.labels_ == -1).sum()}')

---
## 7. Choosing an Algorithm
---

| | k-means | Hierarchical | DBSCAN |
| --- | --- | --- | --- |
| Number of clusters | you choose `k` first | choose a cut height afterwards | an output, not an input |
| Cluster shapes | round, similar sizes | depends on linkage | any shape |
| Can label a point "none" | no | no | yes, as noise |
| Scales to large data | well | poorly | moderately |
| Main difficulty | choosing `k`, initial centroids | choosing linkage and cut | choosing `eps` |

No Free Lunch

**CAUTION.** All three algorithms will return an answer on any dataset you hand them, including data with no group structure at all. k-means asked for three clusters will produce three clusters from pure noise, and the picture will look convincing. There is no error message for "this data has no clusters".

---
## 8. Dimension Reduction with PCA
---
### 8.1 Why reduce dimensions?

The superstore data (https://www.kaggle.com/code/arshmankhalid/personalities-of-customers-discovering-consumer) has six columns recording how much each customer spent on wines, fruit, meat, fish, sweets, and gold. Six dimensions cannot be plotted, and Module 5's pairplot would need 15 panels.

There is a deeper problem than plotting, usually called the **curse of dimensionality**: as the number of features grows, points spread out, distances between them become more and more alike, and "nearest neighbour" stops meaning much. Every algorithm in this module rests on distance, so this matters.

**Principal Component Analysis (PCA)** replaces the original features with new ones, called **principal components**, built as combinations of the originals. The first component is chosen to capture as much of the variation in the data as possible; the second captures as much of what is left as possible, and so on. Keeping the first two or three often preserves most of the information while making the data drawable.

**One important rule: PCA must be given scaled data.** PCA chases variance, and variance depends on units. `Income` measured in dollars has a variance thousands of times larger than `NumWebPurchases`, so without scaling the first component would simply *be* income. Standardizing every column to mean 0 and standard deviation 1 puts them on equal footing.

<img src="https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/ClassNotebooks/09-unsupervised_learning/fig_dimensionality_reduction_techniques.png?raw=true" width="620">

*PCA is one route through a larger family. We use it because it is the most widely applied, and because has a really cool connection to Linear Algebra.*

## Warm-up Example 

Suppose you are invited to a party that you know no one at and are given the following list of attendees:

|  Name | Height | Weight | Age | Hometown |
| --- | --- | --- | --- | --- |
| Alex | 5'6'' | 180 | 30 | MSP |
| Bob | 5'11'' | 181 | 25 | MAD |
| Carl | 6'4'' | 182 | 35 | LX |
| Doug | 5'11'' | 180 | 65 | EC |

How do we know who is who?


In [ ]:
# RUN-TOGETHER
superstore = pd.read_csv('https://raw.githubusercontent.com/jppeirce/DSC210-Foundations-of-Data-Science/main/csv_data/superstore_data.csv')
print('shape:', superstore.shape)

spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
              'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
spend = superstore[spend_cols]
spend.describe().round(1)

### 8.2 The idea in two dimensions

Before we reduce six columns, we watch PCA reduce **two**, where we can see everything.

**Variance is information.** Suppose you had to tell the people in a room apart using a single number. If their heights range from 150 to 190 cm but their weights are all within a kilogram of each other, height does the job and weight adds almost nothing: you could drop weight and lose very little. The measurement that *spreads out more* carries more of the information that tells the rows apart.

After scaling, though, every column has variance exactly 1, so no column can be dropped for free. When two scaled columns are **correlated**, PCA does something better than dropping one: it **combines** them. It finds the direction in which the cloud of points is most stretched (PC1), then the direction perpendicular to it (PC2), and describes every point by its position along those new axes.

The cell below makes up a small dataset of 150 heights and weights (these are simulated, not real people) and shows PCA in three steps. You do not need to follow the plotting code; read the three pictures.

In [ ]:
# RUN-TOGETHER  (made-up data: 150 simulated heights and weights)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

rng = np.random.default_rng(210)
height = rng.normal(170, 9, size=150)
weight = 0.9 * (height - 170) + 70 + rng.normal(0, 4.5, size=150)
toy = pd.DataFrame({'height_cm': height, 'weight_kg': weight})

toy_scaled = StandardScaler().fit_transform(toy)
pca_toy = PCA(n_components=2).fit(toy_scaled)
toy_pcs = pca_toy.transform(toy_scaled)

r = toy['height_cm'].corr(toy['weight_kg'])
print('correlation of height and weight:', round(r, 3))
print('variance explained (PC1, PC2)  :', pca_toy.explained_variance_ratio_.round(3))
print(pd.DataFrame(pca_toy.components_.T, index=toy.columns, columns=['PC1', 'PC2']).round(2))

# ---- three pictures ----
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(toy_scaled[:, 0], toy_scaled[:, 1], s=12, alpha=0.6)
for vec, var, name, color in zip(pca_toy.components_, pca_toy.explained_variance_, ['PC1', 'PC2'], ['C3', 'C2']):
    tip = vec * 2 * np.sqrt(var)
    ax.annotate('', xy=tip, xytext=(0, 0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.text(tip[0] * 1.1, tip[1] * 1.1, name, color=color, fontsize=12)
ax.set_title('1. Scaled data and the two PCA directions')
ax.set_xlabel('height (scaled)'); ax.set_ylabel('weight (scaled)')

ax = axes[1]
ax.scatter(toy_pcs[:, 0], toy_pcs[:, 1], s=12, alpha=0.6)
ax.set_title('2. Rotated: the same points on new axes')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

ax = axes[2]
ax.scatter(toy_pcs[:, 0], toy_pcs[:, 1], s=12, color='lightgray')
ax.vlines(toy_pcs[:, 0], toy_pcs[:, 1], 0, color='C3', lw=0.7)
ax.scatter(toy_pcs[:, 0], np.zeros(len(toy_pcs)), s=12)
ax.set_title('3. Keep PC1 only: the red bits are discarded')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

for ax in axes:
    ax.set_xlim(-3.6, 3.6); ax.set_ylim(-3.6, 3.6); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

Reading the three pictures:

1. **Find the directions.** The cloud is stretched along a diagonal, because tall people in this made-up data tend to be heavy. PC1 points along the longest stretch, and PC2 is perpendicular to it. The loadings for PC1 are 0.71 and 0.71: PC1 is an equal mix of height and weight, roughly "overall body size." (With exactly two scaled columns this is always the case: the loadings are $\pm 0.71$, because $0.71^2 + 0.71^2 \approx 1$ and neither column is favored.)
2. **Rotate.** Describing each person by PC1 and PC2 instead of height and weight is just turning the axes. Every point keeps its distance to every other point, so **nothing is lost yet**. What changes is where the spread lives: about 94% of the variance is now along PC1 and only about 6% along PC2.
3. **Drop.** Keeping only PC1 slides each point straight down (or up) onto the PC1 axis. The red segments are the information we throw away, and together they hold that 6% of the variance. Because the cloud is thin in the PC2 direction, one number per person now does almost the work of two.

For two scaled columns there is a tidy formula: the share of variance on PC1 is $(1 + |r|)/2$, where $r$ is their correlation. Here $r \approx 0.88$, so the share is about $(1 + 0.88)/2 = 0.94$, matching the printout.

> **Try it.** In the cell above, change `0.9 * (height - 170)` to `0 * (height - 170)`, so that weight no longer depends on height, and run it again. What happens to the correlation, the variance explained, and picture 1? Use the formula to explain why PCA cannot help when the columns are uncorrelated.

### 8.3 Scale, then reduce: six spending columns

In [ ]:
# RUN-TOGETHER
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaled = StandardScaler().fit_transform(spend)   # mean 0, sd 1 for every column

pca = PCA(n_components=2).fit(scaled)
components = pca.transform(scaled)

print('variance explained by PC1:', round(pca.explained_variance_ratio_[0], 3))
print('variance explained by PC2:', round(pca.explained_variance_ratio_[1], 3))
print('the two together         :', round(pca.explained_variance_ratio_[:2].sum(), 3))

#### How many components to keep?

We asked for two components because two can be drawn. A **scree plot** shows what every component would add: one bar per component for its share of the variance, and a line for the running total.

In [ ]:
# RUN-TOGETHER
pca_all = PCA().fit(scaled)        # keep all six components, just to look
share = pca_all.explained_variance_ratio_
print('share      :', share.round(3))
print('cumulative :', share.cumsum().round(3))

positions = np.arange(1, len(share) + 1)
plt.bar(positions, share, label='this component')
plt.plot(positions, share.cumsum(), marker='o', color='C3', label='cumulative')
plt.title('Scree plot: superstore spending')
plt.xlabel('principal component')
plt.ylabel('share of variance')
plt.xticks(positions)
plt.legend()
plt.show()

Read it like an elbow plot. PC1 alone holds about 56% of the variance; after it, every component adds a similar small amount (about 12%, 12%, 7%, 7%, and 6%). A rule of thumb you will often see is to keep enough components to reach around 80% to 90% of the variance. Here that would take four or five of the six components, which barely reduces anything. The more honest description of this data is *one strong direction plus a lot of smaller, similar ones*. We keep two because two can be drawn.

Two numbers now stand in for six, and they retain most of the variation. What do they *mean*? Read the **loadings**: how much each original feature contributes to each component.

In [ ]:
# RUN-TOGETHER
loadings = pd.DataFrame(pca.components_.T, index=spend_cols, columns=['PC1', 'PC2']).round(2)
print(loadings)

#### How to read a loadings table

Each column of the table is a **recipe** for one component. To get a customer's PC1 score, take each of the customer's *scaled* spending values, multiply by that column's loading, and add:

$$\text{PC1} = 0.37\,z_{\text{wines}} + 0.43\,z_{\text{fruits}} + 0.44\,z_{\text{meat}} + 0.44\,z_{\text{fish}} + 0.42\,z_{\text{sweets}} + 0.34\,z_{\text{gold}}$$

Here $z$ is the scaled value from `scaled`: how many standard deviations the customer sits above (positive) or below (negative) the average customer in that category. A customer who is exactly average in every category has every $z = 0$, so every score is 0.

Four rules for reading the numbers:

1. **Sign.** A positive loading means spending *more* in that category raises the score; a negative loading means spending more *lowers* it.
2. **Size.** The larger the loading (ignoring its sign), the more that category moves the score. A loading near 0 means the category hardly matters for that component.
3. **Shares.** The squared loadings in each column add to 1 (the cell below checks this), so a squared loading shows roughly what share of the component's weight sits on that category.
4. **The overall sign is arbitrary.** Flipping every sign in a column describes the same direction pointing the other way, and every score would change sign. Different software versions can return either. Read the **pattern** within a column (which categories move together, which move opposite), not whether a score happens to be positive.

One consequence of rule 1 that often surprises people: a **negative loading times a below-average value is a positive contribution**, and a positive loading times a below-average value is a negative one.

In [ ]:
# RUN-TOGETHER  (rule 3: the squared loadings in each component add to 1)
print((pca.components_ ** 2).sum(axis=1).round(3))

> **Discuss.** Look at the PC1 column. Are the loadings all the same sign, or mixed? If a customer scores high on PC1, what does that say about their spending? Now do the same for PC2, where the signs are more interesting.
>
> A principal component is not handed to you with a name. Naming it from its loadings is interpretation, and it is where the substantive-expertise circle from Module 1 is needed.

#### Worked example: two customers, one recipe

The two customers below have almost the same PC1 score but opposite PC2 scores. The table for each one shows its scaled values and how much each category adds to each score.

In [ ]:
# RUN-TOGETHER
z = pd.DataFrame(scaled, columns=spend_cols)     # the scaled values, as a table

for i in [152, 762]:
    print(f'customer {i} spent:', spend.loc[i].tolist(), '| total:', spend.loc[i].sum())
    parts = pd.DataFrame({'z': z.loc[i],
                          'PC1 part': pca.components_[0] * z.loc[i],
                          'PC2 part': pca.components_[1] * z.loc[i]})
    print(parts.round(2))
    print('PC1 by hand =', round(parts['PC1 part'].sum(), 2),
          '  PC2 by hand =', round(parts['PC2 part'].sum(), 2),
          '  PCA says:', components[i].round(2), '\n')

Reading the two tables:

- **Customer 152** spent \$1296 on wine, almost 3 standard deviations above average ($z = 2.95$). Wine has a positive loading on *both* components, so that one category adds 1.10 to PC1 and 1.82 to PC2. Result: PC1 $\approx$ 1.81, PC2 $\approx$ 1.57.
- **Customer 762** spent \$151 on fruit ($z = 3.14$) and \$153 on fish ($z = 2.11$). Those raise PC1 (by 1.35 and 0.93), but fruit and fish have *negative* PC2 loadings, so they pull PC2 down (by 1.07 and 0.60). This customer also spent less than average on wine ($z = -0.72$), and a below-average value times wine's positive PC2 loading lowers PC2 by another 0.44. Result: PC1 $\approx$ 1.93, PC2 $\approx$ -1.95.

So **PC1** behaves like *how much more than typical a customer spends, across the board*, and **PC2** describes the *mix*: wine and gold at one end, fruit, fish, and sweets at the other. That is a naming, and naming is interpretation.

Notice one surprise. Customer 762 spent \$576 in total, a third of customer 152's \$1789 and slightly less than the average customer (about \$606), yet has the *higher* PC1 score. PCA works in standard deviations, not dollars: \$151 is a lot to spend on fruit, where most customers spend little, while \$1296 is large but less unusual for wine. A name like "total spending" is close but not exact, which is a good reason to check any name against a few individual rows.

> **Try it.** Change the list to `[24]`. This customer's PC1 and PC2 are both close to 0. Is that because every $z$ is near 0, or because large contributions cancel? Look at the table before deciding.

#### The whole recipe at a glance

Rule 3 says the squared loadings in each component add to 1, so they can be read as **shares**. A heatmap of the squared loadings shows what each component is made of.

In [ ]:
# RUN-TOGETHER
shares = pd.DataFrame(pca.components_ ** 2, index=['PC1', 'PC2'], columns=spend_cols)
sns.heatmap(shares, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1)
plt.title('Squared loadings: what each component is made of')
plt.show()

PC1 spreads its weight almost evenly (0.12 to 0.19 per category), which fits the name "spending across the board." PC2 puts most of its weight on wine (0.38) and gold (0.28). Squaring hides the signs, so read this picture together with the loadings table: on PC2, wine and gold pull one way while fruit, fish, and sweets pull the other.

In [ ]:
# RUN-TOGETHER
# The six spending columns, drawn in two dimensions.
sns.scatterplot(x=components[:, 0], y=components[:, 1], s=8, alpha=0.4)
plt.title('Superstore customers in two principal components')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.show()

The plot puts each of the 2,240 superstore customers at one point. 
- PC1 on the horizontal axis is roughly how much more than a typical customer they spend across all six categories, and 
- PC2 on the vertical axis is the mix of what they buy (wine and gold toward the top, fruit, fish and sweets toward the bottom). 

The points form a sideways fan. 
- About 1,000 light spenders sit in a tight, dense clump at the far left (PC1 below −1), and a thinner tail stretches right to PC1 ≈ 6.5. 
- As PC1 grows, the cloud also opens up vertically: the spread of PC2 goes from about 0.2 in the left clump to about 1.2 among the big spenders. 

**Summary**: customers who buy little all look alike, while customers who spend a lot differ in what they spend it on. That's the structure k-means will try to split up in the next Activity.

#### **Activity 9.5 - PCA and clustering together**

The two halves of this module combine naturally: reduce first, then cluster in the reduced space.

**A.** Run k-means with `k = 3` on `components` (the two-column PCA output) and colour the scatterplot by cluster (`km_pca.labels_`).

In [ ]:
# FILL-IN  (Activity 9.5, Part A)
km_pca = KMeans(n_clusters=3, n_init=10, random_state=0).fit(components)

sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=____, s=8, alpha=0.6, palette='viridis', legend='full')
plt.title('Superstore customers, clustered in PCA space')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='Cluster', markerscale=2)
plt.show()

**B.** k-means hands back numbers (0, 1, 2), not descriptions. A cluster is only useful once you can say in words who is in it, so now work out what each cluster is describing.

1. Run the first cell below. It shows how many customers are in each cluster and the **median** spending per category in each cluster, with a row for all customers as a reference. (We use the median because spending is right-skewed: a few very large spenders pull the mean up, so the median is a better picture of a typical customer.)
2. For each cluster, compare its row to the `all customers` row. Is it above or below the typical customer in every category, or only in some? Which categories stand out most?
3. Give each cluster a short, plain-English name that a marketing team would understand.
4. Check your names against the PCA picture. The second cell prints each cluster's centre in PC1 and PC2. Recall that PC1 is roughly *how much more than typical a customer spends, across the board*, and PC2 is the *mix* (wine and gold toward positive, fruit, fish, and sweets toward negative). Does the cluster with the largest PC1 centre match the cluster you named as the biggest spenders? Does PC2 help explain how the other clusters differ?
5. The second cell also shows medians for columns that k-means **never saw**: income, children at home, and how customers shop. Do these differences fit the names you chose? Be careful with the wording: a cluster *describes* who these customers are; it does not tell us that income or children *cause* the spending.

In [ ]:
# FILL-IN  (Activity 9.5, Part B, steps 1 to 3)
superstore['cluster'] = km_pca.labels_

# how many customers are in each cluster?
print(superstore['cluster'].value_counts().sort_index())

# median spending in each cluster, plus all customers as a reference row
medians = superstore.groupby(____)[spend_cols].median()
medians.loc['all customers'] = spend.median()
medians

In [ ]:
# FILL-IN  (Activity 9.5, Part B, steps 4 and 5)
# each cluster's centre in PCA space
centres = pd.DataFrame(km_pca.cluster_centers_, columns=['PC1', 'PC2']).round(2)
print(centres, '\n')

# columns that were NOT used to build the clusters
other_cols = ['Income', 'Kidhome', 'Teenhome',
              'NumWebPurchases', 'NumStorePurchases', 'NumDealsPurchases']
superstore.groupby('cluster')[____].median()

**C.** Build an elbow plot for this data (k-means on `components`, for `k` from 1 to 8). Where does the curve bend? Is the bend sharp or gradual? Does it suggest 3, and would you be comfortable defending a different choice?

In [ ]:
# FILL-IN  (Activity 9.5, Part C)
ks = range(1, 9)
wcss_pca = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(____).inertia_ for k in ks]

for k, w in zip(ks, wcss_pca):
    print(f'k = {k}   WCSS = {w:7.1f}')

sns.lineplot(x=list(ks), y=wcss_pca, marker='o')
plt.title('Elbow plot: superstore customers in PCA space')
plt.xlabel('k (number of clusters)')
plt.ylabel('WCSS (inertia)')
plt.xticks(list(ks))
plt.show()

**D.** Redo part A on the **unscaled** `spend` data instead of `scaled`: run PCA on the dollar amounts, then k-means with `k = 3` on the result. Compare the two pictures side by side. (The colours in the two panels are separate labels, so the same colour does not mean the same group.)

Which single original column do you think is dominating, and why? Check your guess against the printed output: the squared loadings show what each unscaled component is made of, and the standard deviations show how spread out each column is in dollars. Then look at the cluster medians for the unscaled version. Which column do the three groups differ on most?

In [ ]:
# FILL-IN  (Activity 9.5, Part D)
pca_raw = PCA(n_components=2).fit(____)            # unscaled dollars this time
components_raw = pca_raw.transform(____)

km_raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(components_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=km_pca.labels_,
                s=8, alpha=0.6, palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Scaled spending (Part A)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

sns.scatterplot(x=components_raw[:, 0], y=components_raw[:, 1], hue=____,
                s=8, alpha=0.6, palette='viridis', legend=False, ax=axes[1])
axes[1].set_title('Unscaled spending, in dollars')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.show()

print('variance explained (unscaled PC1, PC2):', pca_raw.explained_variance_ratio_.round(3), '\n')
print('squared loadings, unscaled:')
print(pd.DataFrame(pca_raw.components_.T ** 2, index=spend_cols, columns=['PC1', 'PC2']).round(2), '\n')
print('standard deviation of each column, in dollars:')
print(spend.std().round(1), '\n')
print('median spending in each unscaled cluster:')
print(superstore.groupby(km_raw.labels_)[spend_cols].median())

**E.** In two sentences: what would you tell the store's marketing team? Name the number of customer groups you found and describe each one in a few words (use your names from Part B), and give one honest caution about that number. The cell below collects the numbers you are most likely to quote.

In [ ]:
# RUN-TOGETHER  (Activity 9.5, Part E: numbers for your write-up)
total_spend = superstore[spend_cols].sum(axis=1)

summary = pd.DataFrame({
    'customers': superstore['cluster'].value_counts().sort_index(),
    'share of customers': superstore['cluster'].value_counts(normalize=True).sort_index().round(2),
    'median total spend': total_spend.groupby(superstore['cluster']).median(),
})
summary

*Part E, your two sentences:*

---
## Adding to the Toolkit
---

Every module from here to the final adds rows to one running table. By the last week you will have the whole collection, and the review will be about **choosing** from it: given the feature types you have and the question you were asked, which tool is right?

The columns never change:

| Tool | Question it answers | Feature types it needs | Label needed? | Key assumption | How it fails |
| --- | --- | --- | --- | --- | --- |
| **k-means** | Are there groups, if I commit to how many? | interval or ratio, **scaled** | **no** | clusters are round and similarly sized | needs `k` in advance; sensitive to starting centroids; fails on crescents |
| **Hierarchical clustering** | What is the nested group structure, at every scale at once? | anything with a distance; **scaled** if numeric | **no** | the linkage rule matches what you mean by "close" | choice of linkage changes the tree; scales poorly to large data |
| **DBSCAN** | Are there dense groups, and which points belong to none? | interval or ratio, **scaled** | **no** | clusters are dense regions separated by sparse ones | `eps` is hard to choose; struggles when densities differ |
| **PCA** | Can I describe many numeric columns with a few? | several interval or ratio, **scaled** | **no** | variance is the thing worth keeping; structure is linear | components are unnamed until you read the loadings; dropping components shrinks distances, so neighbors in a PCA picture can mislead |

**Two columns to notice.** Every row says **no** under "Label needed?", which is what makes this module unsupervised, and every row demands **scaled** features, because all four tools are built on distance or variance and both are unit-sensitive. If you remember one thing from this module when choosing a tool, remember to ask whether your features are on comparable scales.

## Suggested Exercises

1. Six points lie on a line at $x = 2,\ 3,\ 4,\ 10,\ 11,\ 12$, and you run k-means with $k = 2$ starting from $c_1 = 2$ and $c_2 = 3$.

    a. Carry out the assign-and-update steps by hand until convergence, recording the WCSS after each update.

    b. How many iterations were needed?

    c. Compute the WCSS of the clustering $\{2, 3, 4, 10\}$, $\{11, 12\}$. Is it larger or smaller than what k-means found? What does that tell you about the answer k-means reached?

    d. Explain why WCSS can never increase from one iteration to the next.

2. A dendrogram built by single linkage on points P, Q, R, S shows merges at heights 1 (P and Q), 2 (R and S), and 6 (the two pairs).

    a. How many clusters do you get cutting at height 1.5? At 4? At 7?

    b. The last merge happens at height 6 while the first two happen at 1 and 2. What does that gap suggest about the structure of this data?

    c. You are told the true distance from Q to R is 6.5. Is that consistent with the dendrogram under single linkage? Explain.

    d. Give one advantage of hierarchical clustering over k-means, and one disadvantage.

3. A colleague runs DBSCAN on 5,000 sensor readings with `eps=0.5, min_samples=10` and reports "one cluster of 4,900 points, plus 100 noise points."

    a. Give two different explanations for a single enormous cluster.

    b. Which parameter would you change first, and in which direction?

    c. Your colleague suggests switching to k-means with `k=4` instead, "so that nothing gets thrown away as noise." Give one reason this could be the wrong fix.

    d. Name one situation where labelling points as noise is the feature you actually want.

4. You run PCA on eight columns of student data and find PC1 explains 78% of the variance, with loadings that are large and positive on all four exam-score columns and near zero on the other four.

    a. Propose a name for PC1 and justify it from the loadings.

    b. Someone asks whether PC1 "is" the student's average exam score. Is that right? Explain the relationship carefully.

    c. You forgot to scale before running PCA. One column was recorded in dollars and ranged from 0 to 40,000. Predict what PC1 would have looked like, and why.

    d. PC1 through PC3 explain 94% of the variance. State one thing you gain and one thing you lose by working with those three columns instead of the original eight.